# Module

In [ ]:
"""
Main script for training and evaluating temporal GNN models for node classification tasks.
"""

import os
import sys
import time
import json
import shutil
import logging
import warnings
from collections import defaultdict
import pickle

import torch
import torch.nn as nn
import numpy as np
from tqdm import tqdm

# Suppress unnecessary warnings
warnings.filterwarnings("ignore")
logging.getLogger('matplotlib').setLevel(logging.WARNING)

from models.TGAT import TGAT
from models.MemoryModel import MemoryModel, compute_src_dst_node_time_shifts
from models.CAWN import CAWN
from models.TCL import TCL
from models.GraphMixer import GraphMixer
from models.DyGFormer import DyGFormer
from models.modules import MergeLayer, MLPClassifier

from utils.utils import (
    set_random_seed, convert_to_gpu, get_parameter_sizes, create_optimizer,
    get_neighbor_sampler
)
from utils.DataLoader import get_idx_data_loader, get_node_classification_data
from utils.metrics import get_node_classification_metrics
from utils.EarlyStopping import EarlyStopping
from utils.load_configs import get_node_classification_args
from evaluate_models_utils import evaluate_model_node_classification





# Settings

In [ ]:
class Args:
    """
    Configuration class for model and training setup.
    """

    #### Dataset selection ####

    # Real-world datasets
    #dataset_name = "brain"
    dataset_name = "school"
    #dataset_name = "stock"

    # Synthetic dataset
    #dataset_name = "synthetic_exp2.1"
    #dataset_name = "synthetic_exp2.2"
    #### Model selection ####

    # Choose one model to activate
    #model_name = 'JODIE'
    model_name = 'DyRep'
    #model_name = 'TGN'
    #model_name = "TGAT"
    #model_name = 'DyGFormer'

    #### General training configuration ####

    batch_size = 128  # Number of samples per training batch

    gpu = 0  # GPU index if using CUDA, otherwise set -1 for CPU

    #### Neighbor sampling ####

    num_neighbors = 5  # Number of temporal neighbors to sample
    sample_neighbor_strategy = 'uniform'  # Strategy for sampling neighbors
    time_scaling_factor = 10.0  # Scaling factor for time-based features

    #### Model-specific hyperparameters ####

    num_walk_heads = 4  # For models with random walk or multi-head attention
    num_heads = 4  # Number of attention heads
    num_layers = 2  # Number of layers (e.g., GNN or transformer layers)
    walk_length = 5  # Length of random walks (if applicable)
    time_gap = 1.0  # Time gap resolution
    time_feat_dim = 32  # Dimension of time features
    position_feat_dim = 32  # Dimension of position features

    patch_size = 16  # Patch size for temporal sequences    
    channel_embedding_dim = 12  # Embedding size per channel
    
    ### brain,stock
    max_input_sequence_length = 10  # Max length of input temporal sequence
    ### Else
    #max_input_sequence_length = 6  # Max length of input temporal sequence
    #### Training settings ####

    learning_rate = 0.001  # Optimizer learning rate
    dropout = 0.1  # Dropout probability
    num_epochs = 50  # Total number of training epochs
    optimizer = 'Adam'  # Optimizer choice
    weight_decay = 0.0005  # L2 regularization weight
    patience = 5  # Early stopping patience (epochs without improvement)

    #### Dataset splits ####

    # There are no valdiation or test set becuase we simply want to see the node embeddings
    val_ratio = 0.0  # Validation set ratio
    test_ratio = 0.0  # Test set ratio

    #### Experiment settings ####

    num_runs = 1  # Number of training runs for averaging
    negative_sample_strategy = 'random'  # Strategy for negative sampling
    load_best_configs = False  # Whether to load best hyperparameters

    #### Device setup ####

    device = 'cuda:0' if torch.cuda.is_available() and gpu >= 0 else 'cpu'

# Instantiate the args object
args = Args()


# Load Model

In [ ]:
%%time
warnings.filterwarnings('ignore')

node_raw_features, edge_raw_features, full_data, train_data, val_data, test_data = \
    get_node_classification_data(dataset_name=args.dataset_name, val_ratio=args.val_ratio, test_ratio=args.test_ratio)

data = full_data

sorted_indices = np.argsort(data.node_interact_times)
data.src_node_ids = data.src_node_ids[sorted_indices]
data.dst_node_ids = data.dst_node_ids[sorted_indices]
data.node_interact_times = data.node_interact_times[sorted_indices]
data.edge_ids = data.edge_ids[sorted_indices]

full_neighbor_sampler = get_neighbor_sampler(
    data=full_data,
    sample_neighbor_strategy=args.sample_neighbor_strategy,
    time_scaling_factor=args.time_scaling_factor,
    seed=1
)

train_idx_data_loader = get_idx_data_loader(list(range(len(train_data.src_node_ids))), batch_size=args.batch_size, shuffle=False)
val_idx_data_loader = get_idx_data_loader(list(range(len(val_data.src_node_ids))), batch_size=args.batch_size, shuffle=False)
test_idx_data_loader = get_idx_data_loader(list(range(len(test_data.src_node_ids))), batch_size=args.batch_size, shuffle=False)

val_metric_all_runs, test_metric_all_runs = [], []

for run in range(args.num_runs):
    set_random_seed(seed=run)

    args.seed = run
    args.load_model_name = f'{args.model_name}_seed{args.seed}'
    args.save_model_name = f'node_classification_{args.model_name}_seed{args.seed}'

    logging.basicConfig(level=logging.INFO)
    logger = logging.getLogger()
    logger.setLevel(logging.DEBUG)
    os.makedirs(f"./logs/{args.model_name}/{args.dataset_name}/{args.save_model_name}/", exist_ok=True)
    fh = logging.FileHandler(f"./logs/{args.model_name}/{args.dataset_name}/{args.save_model_name}/{str(time.time())}.log")
    fh.setLevel(logging.DEBUG)
    ch = logging.StreamHandler()
    ch.setLevel(logging.WARNING)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    fh.setFormatter(formatter)
    ch.setFormatter(formatter)
    logger.addHandler(fh)
    logger.addHandler(ch)

    run_start_time = time.time()
    logger.info(f"********** Run {run + 1} starts. **********")
    logger.info(f'configuration is {args}')

    if args.model_name == 'TGAT':
        dynamic_backbone = TGAT(node_raw_features, edge_raw_features, full_neighbor_sampler, args.time_feat_dim,
                                    args.num_layers, args.num_heads, args.dropout, args.device)
    elif args.model_name in ['JODIE', 'DyRep', 'TGN']:
        src_node_mean_time_shift, src_node_std_time_shift, dst_node_mean_time_shift_dst, dst_node_std_time_shift = \
                compute_src_dst_node_time_shifts(train_data.src_node_ids, train_data.dst_node_ids, train_data.node_interact_times)
        dynamic_backbone = MemoryModel(node_raw_features, edge_raw_features, full_neighbor_sampler, args.time_feat_dim,
                                           args.model_name, args.num_layers, args.num_heads, args.dropout,
                                           src_node_mean_time_shift, src_node_std_time_shift,
                                           dst_node_mean_time_shift_dst, dst_node_std_time_shift, args.device)
    elif args.model_name == 'CAWN':
        dynamic_backbone = CAWN(node_raw_features, edge_raw_features, full_neighbor_sampler, args.time_feat_dim,
                                    args.position_feat_dim, args.walk_length, args.num_walk_heads, args.dropout, args.device)
    elif args.model_name == 'TCL':
        dynamic_backbone = TCL(node_raw_features, edge_raw_features, full_neighbor_sampler, args.time_feat_dim,
                                   args.num_layers, args.num_heads, args.num_neighbors + 1, args.dropout, args.device)
    elif args.model_name == 'GraphMixer':
        dynamic_backbone = GraphMixer(node_raw_features, edge_raw_features, full_neighbor_sampler, args.time_feat_dim,
                                          args.num_neighbors, args.num_layers, args.dropout, args.device)
    elif args.model_name == 'DyGFormer':
        dynamic_backbone = DyGFormer(node_raw_features, edge_raw_features, full_neighbor_sampler, args.time_feat_dim,
                                         args.channel_embedding_dim, args.patch_size, args.num_layers, args.num_heads,
                                         args.dropout, args.max_input_sequence_length, args.device)
    else:
        raise ValueError(f"Wrong value for model_name {args.model_name}!")

    link_predictor = MergeLayer(node_raw_features.shape[1], node_raw_features.shape[1],
                                    node_raw_features.shape[1], 1)
    model = nn.Sequential(dynamic_backbone, link_predictor)

    load_model_folder = f"./saved_models/{args.model_name}/{args.dataset_name}/{args.load_model_name}"
    early_stopping = EarlyStopping(patience=0, save_model_folder=load_model_folder,
                                       save_model_name=args.load_model_name, logger=logger, model_name=args.model_name)


    # === Load full model state_dict ===
    checkpoint_path = os.path.join(load_model_folder, f"{args.load_model_name}.pkl")
    model.load_state_dict(torch.load(checkpoint_path, map_location=args.device))

    # === Move to correct device ===
    model = convert_to_gpu(model, device=args.device)

    # === Extract dynamic backbone (MemoryModel) ===
    model = model[0]  # type: MemoryModel

    # === Reset memory (crucial before inference) ===
    if args.model_name in ['JODIE', 'DyRep', 'TGN']:
        model.memory_bank.__init_memory_bank__()



# Calculate node embeddings

In [ ]:
%%time
# === Group interactions by integer time (once) ===
interactions_by_time = defaultdict(list)
for i in range(len(data.src_node_ids)):
    t = int(data.node_interact_times[i])
    interactions_by_time[t].append(i)

# === Extract embeddings per time step ===
all_embeddings = []
unique_times = sorted(interactions_by_time.keys())


for t in tqdm(unique_times, desc="Time points"):
    node_to_embs = defaultdict(list)

    for idx in interactions_by_time[t]:
        src = data.src_node_ids[idx]
        dst = data.dst_node_ids[idx]
        time = data.node_interact_times[idx]
        edge_id = data.edge_ids[idx]

        src_np = np.array([src])
        dst_np = np.array([dst])
        t_np = np.array([time])
        e_np = np.array([edge_id])

        with torch.no_grad():
            if args.model_name in ['JODIE', 'DyRep', 'TGN']:
                src_emb, dst_emb = model.compute_src_dst_node_temporal_embeddings(
                    src_node_ids=src_np,
                    dst_node_ids=dst_np,
                    node_interact_times=t_np,
                    edge_ids=e_np,
                    edges_are_positive=True,
                    num_neighbors=args.num_neighbors
                )
            elif args.model_name in ['TGAT', 'CAWN', 'TCL', 'GraphMixer']:
                src_emb, dst_emb = model.compute_src_dst_node_temporal_embeddings(
                    src_node_ids=src_np,
                    dst_node_ids=dst_np,
                    node_interact_times=t_np,
                    num_neighbors=args.num_neighbors
                )
            elif args.model_name == 'DyGFormer':
                src_emb, dst_emb = model.compute_src_dst_node_temporal_embeddings(
                    src_node_ids=src_np,
                    dst_node_ids=dst_np,
                    node_interact_times=t_np
                )
            else:
                raise ValueError(f"Unsupported model {args.model_name} in embedding extraction.")

        node_to_embs[src].append(src_emb.squeeze(0).cpu())
        node_to_embs[dst].append(dst_emb.squeeze(0).cpu())

        if args.model_name in ['JODIE', 'DyRep', 'TGN']:
            model.memory_bank.detach_memory_bank()

    # Average per node
    for node_id, embs in node_to_embs.items():
        avg_emb = torch.stack(embs).mean(dim=0).numpy()
        all_embeddings.append({
            'time': int(t),
            'node_id': int(node_id),
            'embedding': avg_emb
        })

# === Save ===
output_path = f"node_embeddings_discrete_avg_{args.model_name}_{args.dataset_name}.pkl"
with open(output_path, "wb") as f:
    pickle.dump(all_embeddings, f)

    
    
#### ADDED ####
# Sort embeddings first by time, then by node_id
sorted_embeddings = sorted(all_embeddings, key=lambda e: (e['time'], e['node_id']))
text_output_path = f"node_embeddings_discrete_avg_{args.model_name}_{args.dataset_name}.txt"

with open(text_output_path, "w", encoding="utf-8") as f_txt:
    num_nodes = len(set((e['node_id'], e['time']) for e in all_embeddings))
    num_timesteps = len(set(e['time'] for e in all_embeddings))
    f_txt.write(f"{num_nodes} {num_timesteps}\n")
    for entry in sorted_embeddings:
        emb_str = " ".join([f"{x:.6f}" for x in entry['embedding']])
        f_txt.write(f"{entry['node_id']} {emb_str}\n")

print(f"\n? Saved {len(all_embeddings)} node-time embeddings to: {output_path}")
print(f"\n? Saved {len(all_embeddings)} node-time embeddings text to: {text_output_path}")
